# LangGraph：原生 module 與 I/O

本教學直接組裝 `StateGraph`。所有節點與工具的程式碼都顯示在下方。
只定義框架需要的節點 callback 與工具函式。圖形建立與呼叫直接放在區塊中。

| 層級 | 原生 API | 輸入 | 輸出 |
|---|---|---|---|
| 模型 | `langchain_openai.ChatOpenAI.ainvoke` | 訊息清單 | `AIMessage` |
| 工具 | `langchain_core.tools.tool` 產生的工具 `.invoke` | 參數 `dict` | 函式回傳值 |
| 圖形 | `langgraph.graph.StateGraph.compile` | 節點與邊 | `CompiledStateGraph` |
| 執行 | `CompiledStateGraph.ainvoke` | 狀態 `dict` | 狀態 `dict` |
| 串流 | `CompiledStateGraph.astream` | 狀態與串流模式 | 非同步迭代器 |

這份範例的狀態使用 `MessagesState`，所以主要欄位是 `messages`。
圖形的輸出由狀態 schema 決定。LangGraph 沒有固定的 `output` 文字欄位。

本檔已保存一次真實執行的輸出。輸出中的識別碼、時間與用量只代表該次執行。


### 類別關係圖

箭頭表示建立、使用或回傳關係。此圖不是繼承樹。`StateGraph` 是 builder；`compile()` 才產生可執行圖形。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [1]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">類別關係圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 475" role="img" aria-label="LangGraph 類別關係：builder、圖形、模型、狀態與工具訊息" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>LangGraph 類別關係：builder、圖形、模型、狀態與工具訊息</title><defs><marker id="langgraph-agent-classes-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="475" fill="#f8fafc"/><rect x="25" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M26 73 H314" stroke="#dce4ef"/><text x="41" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">StateGraph</text><text x="41" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langgraph.graph</text><text x="41" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">add_node() / add_edge()</text><text x="41" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">compile()</text><rect x="395" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M396 73 H684" stroke="#dce4ef"/><text x="411" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">CompiledStateGraph</text><text x="411" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langgraph.graph.state</text><text x="411" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">ainvoke(input, config)</text><text x="411" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">astream(input, ...)</text><rect x="765" y="25" width="290" height="158" rx="12" fill="#fff" stroke="#2563eb" stroke-width="1.6"/><path d="M766 73 H1054" stroke="#dce4ef"/><text x="781" y="54" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">ChatOpenAI</text><text x="781" y="96" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langchain_openai</text><text x="781" y="125" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">bind_tools([...])</text><text x="781" y="147" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">ainvoke(messages) → AIMessage</text><rect x="25" y="285" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M26 333 H314" stroke="#dce4ef"/><text x="41" y="314" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">MessagesState</text><text x="41" y="356" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langgraph.graph · TypedDict</text><text x="41" y="385" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">messages: list[AnyMessage]</text><text x="41" y="407" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">訊息 reducer 合併節點更新</text><rect x="395" y="285" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M396 333 H684" stroke="#dce4ef"/><text x="411" y="314" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">ToolNode</text><text x="411" y="356" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langgraph.prebuilt</text><text x="411" y="385" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">執行 multiply 工具</text><text x="411" y="407" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">回傳 ToolMessage</text><rect x="765" y="285" width="290" height="158" rx="12" fill="#fff" stroke="#087e8b" stroke-width="1.6"/><path d="M766 333 H1054" stroke="#dce4ef"/><text x="781" y="314" text-anchor="start" fill="#172b4d" font-size="18" font-weight="650">AIMessage / ToolMessage</text><text x="781" y="356" text-anchor="start" fill="#52647c" font-size="12" font-weight="400">langchain_core.messages</text><text x="781" y="385" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">tool_calls / tool_call_id</text><text x="781" y="407" text-anchor="start" fill="#172b4d" font-size="14" font-weight="400">content 與 metadata</text><polyline points="315,104 395,104" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-classes-arrow)"/><text x="355.0" y="90" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">compile()</text><polyline points="685,104 765,104" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-classes-arrow)"/><text x="725.0" y="90" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">節點呼叫</text><polyline points="460,183 460,230 170,230 170,285" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-classes-arrow)"/><text x="300" y="218" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">使用此狀態</text><polyline points="540,183 540,285" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-classes-arrow)"/><text x="606" y="238" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">執行工具節點</text><polyline points="910,183 910,285" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-classes-arrow)"/><text x="975" y="238" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">回傳訊息</text><polyline points="685,365 765,365" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-classes-arrow)"/><text x="725.0" y="350" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">回傳</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">箭頭表示建立、使用或回傳關係。此圖不是繼承樹。StateGraph 是 builder；compile() 才產生可執行圖形。</p>
</section>
"""))


## 1. 啟動與設定

在本資料夾執行：

```bash
uv sync --locked --group notebook
uv run --group notebook jupyter lab tutorial_native.ipynb
```

依序執行各區塊。推論與串流區塊會呼叫真實模型，並使用模型額度。
本教學直接使用頂層 `await`。每個串流範例會啟動另一個任務。
所有原生輸入與輸出都保留在 Notebook 變數中。


In [1]:
import os
import sys
from importlib.metadata import version
from pathlib import Path
from pprint import pprint

from dotenv import dotenv_values

assert sys.version_info[:2] == (3, 13), "請使用 Python 3.13 核心。"
project_dir = Path.cwd()
assert (project_dir / "pyproject.toml").is_file(), "請從本專案資料夾啟動 Notebook。"

# 直接載入設定。環境變數優先於專案 .env，再優先於根目錄 .env。
env = {
    **dotenv_values(project_dir.parent / ".env"),
    **dotenv_values(project_dir / ".env"),
    **os.environ,
}
endpoint = env["LITELLM_ENDPOINT"].strip().rstrip("/")
api_key = env["LITELLM_API_KEY"]
model_name = env["LITELLM_MODEL"]
assert endpoint.startswith(("http://", "https://")), "端點需要 HTTP 或 HTTPS 協定。"
assert api_key and model_name, "請設定金鑰與模型別名。"
print("Python：", sys.version.split()[0])
print("模型：", model_name)
# 請勿顯示 env、api_key 或含有金鑰的 client 設定。

Python： 3.13.15
模型： gpt-4.1-mini


## 2. 直接建立模型

LangGraph 負責流程。此範例的模型介面由 `langchain_openai` 提供。
`ChatOpenAI` 透過 LiteLLM 的標準 Chat Completions API 連線。


In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=model_name,
    base_url=endpoint,
    api_key=api_key,
    use_responses_api=False,
    stream_usage=False,
    timeout=60,
    max_retries=0,
)
print(type(model))
print("langchain-openai：", version("langchain-openai"))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
langchain-openai： 1.7.3


## 3. 觀察模型本身的 I/O

傳入 `SystemMessage` 與 `HumanMessage` 的清單。原生回傳值是 `AIMessage`。
先檢查物件，再檢查其 Pydantic 序列化結果。
`model_dump()` 是 SDK 物件本身的方法。


In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

model_input = [
    SystemMessage(content="以繁體中文回答。使用一句短句。"),
    HumanMessage(content="說明 agent 的用途。"),
]
pprint(model_input)
model_output = await model.ainvoke(model_input)
print(type(model_output))
pprint(model_output)
pprint(model_output.model_dump())

[SystemMessage(content='以繁體中文回答。使用一句短句。', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='說明 agent 的用途。', additional_kwargs={}, response_metadata={})]


<class 'langchain_core.messages.ai.AIMessage'>
AIMessage(content='Agent 用於自動執行任務與決策。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 28, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_3e00b34037', 'id': 'chatcmpl-EgAlI1lOVlAoupqG0xvdGQU9aH7FK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a208-db42-751b-9310-dbfde4c3dfbc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 14, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r

`content` 是訊息內容。它可以是文字，也可以是內容區塊清單。
`tool_calls` 是模型提出的工具請求。它不代表工具已執行。
`usage_metadata` 與 `response_metadata` 的內容取決於上游服務。


In [4]:
print("content：", model_output.content)
print("tool_calls：", model_output.tool_calls)
print("usage_metadata：", model_output.usage_metadata)
print("response_metadata：", model_output.response_metadata)

content： Agent 用於自動執行任務與決策。
tool_calls： []
usage_metadata： {'input_tokens': 28, 'output_tokens': 14, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
response_metadata： {'token_usage': {'completion_tokens': 14, 'prompt_tokens': 28, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_3e00b34037', 'id': 'chatcmpl-EgAlI1lOVlAoupqG0xvdGQU9aH7FK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}


## 4. 宣告原生工具

`@tool` 將函式轉成 LangChain 工具。它會根據型別建立參數 schema。
這個函式就是代理可呼叫的工具，所有實作都在此區塊。
直接傳入參數字典時，`.invoke()` 回傳數值。


In [5]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """計算 a 乘 b。"""
    return a * b


print(type(multiply))
pprint(multiply.args_schema.model_json_schema())
tool_input = {"a": 6, "b": 7}
tool_output = multiply.invoke(tool_input)
print("工具輸入：", tool_input)
print("工具輸出：", type(tool_output), tool_output)
model_with_tools = model.bind_tools([multiply])

<class 'langchain_core.tools.structured.StructuredTool'>
{'description': '計算 a 乘 b。',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
                'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'multiply',
 'type': 'object'}
工具輸入： {'a': 6, 'b': 7}
工具輸出： <class 'int'> 42


## 5. 直接組裝 StateGraph

`call_model` 是模型節點。輸入是目前狀態，輸出是狀態更新。
`MessagesState` 會將訊息更新合併至 `messages`。
`ToolNode` 執行工具，並回傳 `ToolMessage`。
`tools_condition` 檢查最後一則訊息的工具請求，再決定執行工具或結束。


In [6]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition


async def call_model(state: MessagesState):
    response = await model_with_tools.ainvoke(
        [
            SystemMessage(content="你是計算助理。乘法必須使用 multiply 工具。以繁體中文回答。"),
            *state["messages"],
        ]
    )
    return {"messages": [response]}


graph_builder = StateGraph(MessagesState)
graph_builder.add_node("model", call_model)
graph_builder.add_node("tools", ToolNode([multiply]))
graph_builder.add_edge(START, "model")
graph_builder.add_conditional_edges("model", tools_condition)
graph_builder.add_edge("tools", "model")
graph = graph_builder.compile()
print(type(graph))
print("langgraph：", version("langgraph"))
print(graph.get_graph().draw_mermaid())

<class 'langgraph.graph.state.CompiledStateGraph'>
langgraph： 1.2.19
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools<hr/><small><em>__tools = multiply</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### 控制流程圖

此圖對應下方的 `StateGraph` 組裝。`tools_condition` 在有工具呼叫時前往 `tools`，否則結束。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [2]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">控制流程圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 600" role="img" aria-label="LangGraph 條件邊與工具迴圈" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>LangGraph 條件邊與工具迴圈</title><defs><marker id="langgraph-agent-flow-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="600" fill="#f8fafc"/><rect x="410" y="20" width="260" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="48" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">輸入狀態</text><text x="540.0" y="71" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">messages: [HumanMessage]</text><rect x="410" y="135" width="260" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="163" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">模型節點</text><text x="540.0" y="186" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">model.ainvoke(messages)</text><polygon points="540,245 690,310 540,375 390,310" fill="#fff7ed" stroke="#b45309" stroke-width="1.5"/><text x="540" y="308" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">有 tool_calls？</text><rect x="765" y="277" width="280" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="905.0" y="305" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">ToolNode</text><text x="905.0" y="328" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">執行 multiply(a, b)</text><rect x="765" y="400" width="280" height="68" rx="12" fill="#f0f6ff" stroke="#087e8b" stroke-width="1.5"/><text x="905.0" y="428" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">ToolMessage</text><text x="905.0" y="451" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">tool_call_id 對應呼叫</text><rect x="410" y="472" width="260" height="68" rx="12" fill="#f0f6ff" stroke="#2563eb" stroke-width="1.5"/><text x="540.0" y="500" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">結束並回傳狀態</text><text x="540.0" y="523" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">messages 含最終 AIMessage</text><polyline points="540,88 540,135" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-flow-arrow)"/><polyline points="540,203 540,245" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-flow-arrow)"/><polyline points="690,310 765,310" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-flow-arrow)"/><text x="727.5" y="295" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">是</text><polyline points="540,375 540,472" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-flow-arrow)"/><text x="565" y="425" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">否</text><polyline points="905,345 905,400" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-flow-arrow)"/><polyline points="1045,434 1062,434 1062,168 670,168" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-flow-arrow)"/><text x="870" y="154" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">回到模型</text><text x="28" y="570" text-anchor="start" fill="#52647c" font-size="14" font-weight="400">每個箭頭表示本例的一次狀態傳遞。模型可執行多輪工具呼叫。</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">此圖對應下方的 StateGraph 組裝。tools_condition 在有工具呼叫時前往 tools，否則結束。</p>
</section>
"""))


## 6. 直接呼叫圖形，檢查完整狀態

`graph_input` 是狀態字典。`config` 是執行設定，與狀態分開傳入。
`recursion_limit` 限制圖形步數。
這個圖形沒有 checkpointer。每次呼叫都從本次傳入的狀態開始。


In [7]:
graph_input = {"messages": [HumanMessage(content="請計算 6 乘 7。")]}
config = {"recursion_limit": 20}
pprint(graph_input)
result = await graph.ainvoke(graph_input, config=config)
print(type(result))
print("狀態欄位：", list(result))
pprint(result)

{'messages': [HumanMessage(content='請計算 6 乘 7。', additional_kwargs={}, response_metadata={})]}


<class 'dict'>
狀態欄位： ['messages']
{'messages': [HumanMessage(content='請計算 6 乘 7。', additional_kwargs={}, response_metadata={}, id='4701b6d9-3112-46b8-af8d-7db5cafff3e9'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 80, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_75e7b04702', 'id': 'chatcmpl-EgAlJ86W3d6LPiCRPY2Vcz4GWbrfa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1a208-e185-7445-820e-99ae0f7a3733-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6, 'b': 7}, 'id': 'call_KETNtm7G7iJ

## 7. 逐則讀取原生 Message

預期可觀察到使用者訊息、模型工具請求、工具結果與模型回答。
`AIMessage.tool_calls` 的 `id` 對應 `ToolMessage.tool_call_id`。
下面直接讀取原生欄位。`result` 仍保留完整狀態。


In [8]:
for message in result["messages"]:
    print(type(message))
    pprint(message.model_dump())
    if isinstance(message, AIMessage):
        print("模型提出的工具呼叫：", message.tool_calls)
    if isinstance(message, ToolMessage):
        print("工具結果對應的呼叫：", message.tool_call_id)

final_message = result["messages"][-1]
print("最後訊息內容：", final_message.content)

<class 'langchain_core.messages.human.HumanMessage'>
{'additional_kwargs': {},
 'content': '請計算 6 乘 7。',
 'id': '4701b6d9-3112-46b8-af8d-7db5cafff3e9',
 'name': None,
 'response_metadata': {},
 'type': 'human'}
<class 'langchain_core.messages.ai.AIMessage'>
{'additional_kwargs': {'refusal': None},
 'content': '',
 'id': 'lc_run--01a1a208-e185-7445-820e-99ae0f7a3733-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'tool_calls',
                       'id': 'chatcmpl-EgAlJ86W3d6LPiCRPY2Vcz4GWbrfa',
                       'logprobs': None,
                       'model_name': 'gpt-4.1-mini',
                       'model_provider': 'openai',
                       'service_tier': 'default',
                       'system_fingerprint': 'fp_75e7b04702',
                       'token_usage': {'completion_tokens': 17,
                                       'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                      

### 循序圖

圖中是一輪工具呼叫。`ToolMessage.tool_call_id` 會對應 `AIMessage.tool_calls` 的呼叫 ID。

執行下方區塊可顯示圖形。此區塊不會呼叫 LLM。


In [3]:
from IPython.display import HTML, display

display(HTML(r"""
<section style="background:#f8fafc;color:#172b4d;border:1px solid #dce4ef;border-radius:16px;padding:18px;margin:12px 0;font-family:system-ui,sans-serif;overflow-x:auto">
<h3 style="color:#172b4d;margin:0 0 12px;font-size:20px">循序圖</h3>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1080 552" role="img" aria-label="LangGraph 一次工具呼叫的循序圖" style="display:block;width:100%;min-width:760px;height:auto;font-family:'Noto Sans CJK TC','Microsoft JhengHei',system-ui,sans-serif">
 <title>LangGraph 一次工具呼叫的循序圖</title><defs><marker id="langgraph-agent-sequence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#64748b"/></marker></defs>
 <rect width="1080" height="552" fill="#f8fafc"/><rect x="16.0" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="110.0" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">Notebook</text><text x="110.0" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">使用者程式</text><line x1="110.0" y1="77" x2="110.0" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="302.6666666666667" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="396.6666666666667" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">CompiledStateGraph</text><text x="396.6666666666667" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">LangGraph runtime</text><line x1="396.6666666666667" y1="77" x2="396.6666666666667" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="589.3333333333334" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="683.3333333333334" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">ChatOpenAI</text><text x="683.3333333333334" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">模型介面</text><line x1="683.3333333333334" y1="77" x2="683.3333333333334" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><rect x="876.0" y="15" width="188" height="62" rx="10" fill="#fff" stroke="#2563eb"/><text x="970.0" y="41" text-anchor="middle" fill="#172b4d" font-size="16" font-weight="600">ToolNode</text><text x="970.0" y="62" text-anchor="middle" fill="#52647c" font-size="11" font-weight="400">工具節點</text><line x1="970.0" y1="77" x2="970.0" y2="502" stroke="#b6c4d6" stroke-dasharray="5 5"/><polyline points="110.0,115 396.6666666666667,115" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)"/><text x="253.33333333333334" y="105" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">1. ainvoke({messages}, config)</text><polyline points="396.6666666666667,164 683.3333333333334,164" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)"/><text x="540.0" y="154" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">2. ainvoke(messages)</text><polyline points="683.3333333333334,213 396.6666666666667,213" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="540.0" y="203" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">3. AIMessage(tool_calls)</text><polyline points="396.6666666666667,262 970.0,262" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)"/><text x="683.3333333333334" y="252" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">4. 執行 multiply(6, 7)</text><polyline points="970.0,311 396.6666666666667,311" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="683.3333333333334" y="301" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">5. ToolMessage(content=&quot;42&quot;)</text><polyline points="396.6666666666667,360 683.3333333333334,360" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)"/><text x="540.0" y="350" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">6. 含工具結果的 messages</text><polyline points="683.3333333333334,409 396.6666666666667,409" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="540.0" y="399" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">7. 最終 AIMessage</text><polyline points="396.6666666666667,458 110.0,458" fill="none" stroke="#64748b" stroke-width="1.8" marker-end="url(#langgraph-agent-sequence-arrow)" stroke-dasharray="6 4"/><text x="253.33333333333334" y="448" text-anchor="middle" fill="#52647c" font-size="13" font-weight="400">8. 狀態 dict：messages</text><text x="30" y="530" text-anchor="start" fill="#52647c" font-size="13" font-weight="400">實線：呼叫或請求　　虛線：回傳資料或事件</text></svg>
<p style="color:#52647c;line-height:1.7;margin:10px 0 0;font-size:14px">圖中是一輪工具呼叫。ToolMessage.tool_call_id 會對應 AIMessage.tool_calls 的呼叫 ID。</p>
</section>
"""))


## 8. 直接接收原生串流

`stream_mode="updates"` 回傳各節點的狀態更新。
此處明確使用 `version="v1"`，每個 chunk 是以節點名稱為鍵的字典。
它不是文字 token 串流，也不是每一步的完整狀態。
下列呼叫啟動另一個計算任務。


In [9]:
updates = []
async for chunk in graph.astream(
    {"messages": [HumanMessage(content="使用工具計算 8 乘 9。")]},
    config={"recursion_limit": 20},
    stream_mode="updates",
    version="v1",
):
    updates.append(chunk)
    print(type(chunk))
    pprint(chunk)

<class 'dict'>
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 81, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_75e7b04702', 'id': 'chatcmpl-EgAlLDU7nZTHp7N03NHsYaYX8m2l2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1a208-e8cf-7381-a081-d30f1437bb28-0', tool_calls=[{'name': 'multiply', 'args': {'a': 8, 'b': 9}, 'id': 'call_mOPk3mGPYqRuU21joI2kktS6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 17, 'total_tokens': 98, 'input_

<class 'dict'>
{'model': {'messages': [AIMessage(content='8 乘 9 的結果是 72。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 106, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_75e7b04702', 'id': 'chatcmpl-EgAlMYaiZEnKbLWMeIXjl6U3d8VgC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a208-ec59-767f-b1da-362941762baf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 12, 'total_tokens': 118, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning

## 官方來源與版本

- [LangGraph Quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)：節點、狀態與工具迴圈。
- [LangGraph 串流](https://docs.langchain.com/oss/python/langgraph/streaming)：`astream` 與 `updates` 模式。
- [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)：模型訊息與工具呼叫。

以專案的 `uv.lock` 為準。此教學使用已安裝版本的公開匯入路徑。
物件顯示的內部 module 路徑可能不同；請使用範例中的公開 import。
